<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 9 — Date Drift Amortization Audit (Google Colab)

This notebook is designed for **beginners** and walks through **Task 9 step by step**.

## What this task is about
LLMs often make mistakes with:
- leap years
- irregular payment dates
- actual day counts
- daily interest calculations

So for this task, we will do **two things**:
1. Compute the **correct amortization schedule with Python** using the exact dates
2. Ask **Gemini** to solve the same problem and compare its answer to the Python ground truth

That way, your notebook shows both:
- the **correct audited answer**
- the **risk of date drift in LLM outputs**


## The loan scenario

We are auditing this loan:

- **Loan Principal:** $100,000
- **Annual Rate:** 12%
- **Daily interest formula:** `Balance * (0.12 / 365) * Days_Elapsed`
- **Start Date:** Jan 1, 2024
- **Payments:**
  - Jan 31: $5,000
  - Feb 29: $5,000
  - Mar 31: $5,000

Important:
- 2024 is a **leap year**
- the formula still says divide by **365**
- interest must be calculated using the **actual number of days elapsed**


In [ ]:
!pip -q install -U google-genai pandas matplotlib

## Step 1 — Add your Gemini API key

Replace `YOUR_API_KEY_HERE` with your real Gemini API key.
If you do not want to use Gemini yet, you can still run the Python audit parts first.


In [ ]:
import os

os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

if os.environ["GEMINI_API_KEY"] == "YOUR_API_KEY_HERE":
    print("⚠️ Gemini API key not added yet. That is okay for now if you only want to run the Python audit first.")
else:
    print("✅ Gemini API key set.")

## Step 2 — Import the Python libraries


In [ ]:
import json
from datetime import datetime
from decimal import Decimal, getcontext
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

getcontext().prec = 28

## Step 3 — Define the loan inputs

You do **not** need to change anything here unless your lecturer gives a different scenario.


In [ ]:
principal = Decimal("100000")
annual_rate = Decimal("0.12")
daily_rate = annual_rate / Decimal("365")
start_date = datetime.strptime("2024-01-01", "%Y-%m-%d").date()

payments = [
    {"date": datetime.strptime("2024-01-31", "%Y-%m-%d").date(), "payment": Decimal("5000")},
    {"date": datetime.strptime("2024-02-29", "%Y-%m-%d").date(), "payment": Decimal("5000")},
    {"date": datetime.strptime("2024-03-31", "%Y-%m-%d").date(), "payment": Decimal("5000")},
]

print("Principal:", principal)
print("Annual rate:", annual_rate)
print("Daily rate:", daily_rate)
print("Start date:", start_date)
print("Payments:")
for row in payments:
    print(" -", row["date"], ":", row["payment"])

## Step 4 — Create the **correct** amortization schedule

This is the most important part of the notebook.

We calculate:
- actual days elapsed between dates
- interest for that exact number of days
- new balance after each payment

Assumption used:
- interest accrues first for the elapsed days
- then the payment is applied on the payment date


In [ ]:
def money(x):
    return Decimal(x).quantize(Decimal("0.01"))

def build_exact_schedule(principal, daily_rate, start_date, payments):
    rows = []
    balance = principal
    previous_date = start_date

    for entry in payments:
        payment_date = entry["date"]
        payment_amount = entry["payment"]

        days_elapsed = (payment_date - previous_date).days
        opening_balance = balance
        interest = opening_balance * daily_rate * Decimal(days_elapsed)
        closing_balance = opening_balance + interest - payment_amount

        rows.append({
            "period_start": previous_date.isoformat(),
            "payment_date": payment_date.isoformat(),
            "days_elapsed": days_elapsed,
            "opening_balance": float(money(opening_balance)),
            "interest": float(money(interest)),
            "payment": float(money(payment_amount)),
            "closing_balance": float(money(closing_balance)),
        })

        balance = closing_balance
        previous_date = payment_date

    return rows, balance

exact_rows, exact_final_balance = build_exact_schedule(
    principal=principal,
    daily_rate=daily_rate,
    start_date=start_date,
    payments=payments
)

df_exact = pd.DataFrame(exact_rows)
df_exact

## Step 5 — Show the exact audited answer


In [ ]:
exact_total_interest = sum(Decimal(str(x["interest"])) for x in exact_rows)

print("Exact total interest charged:", money(exact_total_interest))
print("Exact ending balance after Mar 31 payment:", money(exact_final_balance))

## Step 6 — Build a **naive 30-day schedule**

This is the kind of simplification that often causes date drift:
- assume every period is 30 days
- ignore the actual calendar

This is **not** the correct answer.
It is only here so you can compare it to the exact audited result.


In [ ]:
def build_naive_30_day_schedule(principal, daily_rate, payments, assumed_days=30):
    rows = []
    balance = principal

    for i, entry in enumerate(payments, start=1):
        opening_balance = balance
        interest = opening_balance * daily_rate * Decimal(assumed_days)
        payment_amount = entry["payment"]
        closing_balance = opening_balance + interest - payment_amount

        rows.append({
            "period_number": i,
            "assumed_days": assumed_days,
            "opening_balance": float(money(opening_balance)),
            "interest": float(money(interest)),
            "payment": float(money(payment_amount)),
            "closing_balance": float(money(closing_balance)),
        })

        balance = closing_balance

    return rows, balance

naive_rows, naive_final_balance = build_naive_30_day_schedule(
    principal=principal,
    daily_rate=daily_rate,
    payments=payments,
    assumed_days=30
)

df_naive = pd.DataFrame(naive_rows)
df_naive

## Step 7 — Compare exact vs naive

This tells you how much error a simplified schedule introduces.


In [ ]:
difference = money(naive_final_balance - exact_final_balance)

comparison_df = pd.DataFrame({
    "method": ["Exact actual-day audit", "Naive 30-day assumption"],
    "ending_balance": [float(money(exact_final_balance)), float(money(naive_final_balance))]
})

print(comparison_df)
print("\nDifference (naive - exact):", difference)

## Step 8 — Optional extra comparison: textbook monthly shortcut

Some people incorrectly use:
- monthly rate = 12% / 12 = 1%
- one payment per month
- no actual day counting

That is an even more simplified shortcut.


In [ ]:
def build_monthly_shortcut_schedule(principal, monthly_rate, payments):
    rows = []
    balance = principal

    for i, entry in enumerate(payments, start=1):
        opening_balance = balance
        interest = opening_balance * monthly_rate
        payment_amount = entry["payment"]
        closing_balance = opening_balance + interest - payment_amount

        rows.append({
            "period_number": i,
            "opening_balance": float(money(opening_balance)),
            "interest": float(money(interest)),
            "payment": float(money(payment_amount)),
            "closing_balance": float(money(closing_balance)),
        })

        balance = closing_balance

    return rows, balance

monthly_rows, monthly_final_balance = build_monthly_shortcut_schedule(
    principal=principal,
    monthly_rate=Decimal("0.12") / Decimal("12"),
    payments=payments
)

print("Monthly shortcut ending balance:", money(monthly_final_balance))
print("Monthly shortcut minus exact:", money(monthly_final_balance - exact_final_balance))

## Step 9 — Visual comparison

This chart makes it easier to discuss the audit in your submission.


In [ ]:
plot_df = pd.DataFrame({
    "Method": ["Exact", "Naive 30-day", "Monthly shortcut"],
    "Ending Balance": [
        float(money(exact_final_balance)),
        float(money(naive_final_balance)),
        float(money(monthly_final_balance)),
    ]
})

plt.figure(figsize=(9, 5))
plt.bar(plot_df["Method"], plot_df["Ending Balance"])
plt.ylabel("Ending Balance ($)")
plt.title("Ending Balance Comparison: Exact Audit vs Common Shortcuts")
plt.xticks(rotation=0)
plt.show()

## Step 10 — Connect to Gemini

Now we ask Gemini to solve the same problem.

Important:
- Gemini's answer is **not** the source of truth
- the Python audit is the source of truth
- Gemini is being tested against the exact calculation


In [ ]:
from google import genai

client = None
MODEL_NAME = "gemini-2.5-flash"

if os.environ["GEMINI_API_KEY"] != "GOOGLE_API_KEY":
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    print("✅ Gemini client ready.")
else:
    print("⚠️ Add your API key first if you want to run Gemini cells.")

## Step 11 — Create two prompts

We will test:
1. a **sloppy prompt** that is more likely to produce date drift
2. a **careful audit prompt** that forces exact date handling

Note:
The sloppy prompt may or may not fail every time.
That is normal.
The purpose is to show that prompt quality matters.


In [ ]:
sloppy_prompt = '''
A $100,000 small business loan starts on Jan 1, 2024 at 12% annual interest.
There are three payments of $5,000 on Jan 31, Feb 29, and Mar 31.
Please create an amortization table and give the ending balance.
Keep it simple and concise.
'''

careful_prompt = '''
You are performing a loan audit.

Use this exact formula for each payment period:
Interest = Balance * (0.12 / 365) * Days_Elapsed

Do NOT assume 30 days per month.
Do NOT use a monthly rate shortcut.
Use the actual elapsed days between:
- Jan 1, 2024 and Jan 31, 2024
- Jan 31, 2024 and Feb 29, 2024
- Feb 29, 2024 and Mar 31, 2024

Loan principal: $100,000
Payments:
- Jan 31, 2024: $5,000
- Feb 29, 2024: $5,000
- Mar 31, 2024: $5,000

Return valid JSON only in this format:
{
  "rows": [
    {
      "payment_date": "YYYY-MM-DD",
      "days_elapsed": 0,
      "opening_balance": 0.0,
      "interest": 0.0,
      "payment": 0.0,
      "closing_balance": 0.0
    }
  ],
  "final_balance": 0.0,
  "notes": "short note"
}
'''

## Step 12 — Ask Gemini with the sloppy prompt


In [ ]:
def ask_gemini(prompt):
    if client is None:
        raise ValueError("Gemini client is not ready. Add your API key first.")
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return response.text

if client is not None:
    sloppy_response = ask_gemini(sloppy_prompt)
    print(sloppy_response)
else:
    print("Gemini not configured yet.")

## Step 13 — Ask Gemini with the careful audit prompt


In [ ]:
if client is not None:
    careful_response = ask_gemini(careful_prompt)
    print(careful_response)
else:
    print("Gemini not configured yet.")

## Step 14 — Parse the careful Gemini response

Because we asked for JSON, this response is easier to compare to the Python truth.


In [ ]:
def extract_json_from_text(text):
    text = text.strip()
    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()
    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in response.")
    return json.loads(text[start:end+1])

if client is not None:
    try:
        careful_json = extract_json_from_text(careful_response)
        print("✅ JSON parsed successfully.")
        display(pd.DataFrame(careful_json["rows"]))
        print("Gemini final balance:", careful_json["final_balance"])
    except Exception as e:
        careful_json = None
        print("Could not parse Gemini JSON:", e)
else:
    careful_json = None
    print("Gemini not configured yet.")

## Step 15 — Compare Gemini to the exact audited answer


In [ ]:
if careful_json is not None:
    gemini_final_balance = Decimal(str(careful_json["final_balance"]))
    gemini_error = money(gemini_final_balance - exact_final_balance)

    print("Exact audited ending balance :", money(exact_final_balance))
    print("Gemini ending balance        :", money(gemini_final_balance))
    print("Gemini error vs exact audit  :", gemini_error)
else:
    print("No parsed Gemini result available.")

## Step 16 — Generate a short final audit memo

This creates a clean summary you can paste into your submission.


In [ ]:
exact_final = money(exact_final_balance)
naive_final = money(naive_final_balance)
monthly_final = money(monthly_final_balance)
naive_diff = money(naive_final_balance - exact_final_balance)
monthly_diff = money(monthly_final_balance - exact_final_balance)

if careful_json is not None:
    gemini_final = money(Decimal(str(careful_json["final_balance"])))
    gemini_diff = money(Decimal(str(careful_json["final_balance"])) - exact_final_balance)
    gemini_line = f"Gemini's careful prompt produced an ending balance of ${gemini_final}, which differs from the Python audit by ${gemini_diff}."
else:
    gemini_line = "Gemini was not evaluated in this run because no API key was provided or the JSON response could not be parsed."

final_memo = f'''
Task 9 Audit Memo

Using the exact daily-interest formula Balance * (0.12 / 365) * Days_Elapsed and the actual payment dates,
the correct audited ending balance after the March 31, 2024 payment is ${exact_final}.

The exact day counts are:
- Jan 1 to Jan 31 = 30 days
- Jan 31 to Feb 29 = 29 days
- Feb 29 to Mar 31 = 31 days

This matters because simplified methods produce different answers.
A naive 30-day-per-period approach gives an ending balance of ${naive_final}, which is ${naive_diff} higher than the exact audited result.
A textbook monthly 1% shortcut gives an ending balance of ${monthly_final}, which is ${monthly_diff} higher than the exact audited result.

{gemini_line}

Conclusion:
The correct audit must use the actual elapsed days and the stated daily-interest formula.
Any bank schedule that used smoothed monthly assumptions instead of exact date intervals would be inaccurate.
'''.strip()

print(final_memo)

## Step 17 — Save your outputs


In [ ]:
Path("task9_outputs").mkdir(exist_ok=True)

df_exact.to_csv("task9_outputs/exact_schedule.csv", index=False)
df_naive.to_csv("task9_outputs/naive_30_day_schedule.csv", index=False)

with open("task9_outputs/final_audit_memo.txt", "w", encoding="utf-8") as f:
    f.write(final_memo)

if 'sloppy_response' in globals():
    with open("task9_outputs/gemini_sloppy_response.txt", "w", encoding="utf-8") as f:
        f.write(sloppy_response)

if 'careful_response' in globals():
    with open("task9_outputs/gemini_careful_response.txt", "w", encoding="utf-8") as f:
        f.write(careful_response)

print("✅ Files saved in task9_outputs/")

## Step 18 — Download the final audit memo


In [ ]:
from google.colab import files
files.download("task9_outputs/final_audit_memo.txt")

## Troubleshooting

### If the exact numbers look strange
- Re-run the cells from Step 3 onward
- Make sure you did not change the dates or the formula

### If Gemini returns invalid JSON
- Re-run the prompt cell
- Sometimes models wrap JSON in code fences; the parser already tries to handle that
- If it still fails, print the response and inspect it manually

### If you only want the correct answer for the assignment
That is already produced by the **Python exact audit**.
Gemini is only an additional audit demonstration.
